Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Web search tool

 **One job:** run a DuckDuckGo search from an async context without blocking the event loop.

 `ddgs` is synchronous — same problem as `smtplib` in `email_sender.py`.
 `run_in_executor` offloads it to a thread pool, freeing the event loop immediately.


![web seach diagramm](image/web_search.png)

In [ ]:
import asyncio
import time
from langchain_core.tools import tool
from ddgs import DDGS
from app.core.logging import get_logger

log = get_logger(__name__)

 ## `_ddg_search`

 Synchronous — runs in the thread pool via `run_in_executor`.

 Three attempts with exponential backoff: `sleep(1)` then `sleep(2)`.
 DuckDuckGo rate-limits aggressively under repeated calls.
 On the third failure it re-raises so the caller gets the actual exception.

 `query[:200]` truncates before sending — long LLM-generated queries
 can be rejected by DDG.

In [ ]:
def _ddg_search(query: str) -> list[dict]:
    """Synchronous DDG search with retry — called from run_in_executor."""
    for attempt in range(3):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query[:200], max_results=5))
                if results:
                    return results
        except Exception as e:
            log.warning("ddg_retry: attempt=%s error=%s", attempt + 1, str(e))
            if attempt < 2:
                time.sleep(2 ** attempt)  # 1s then 2s
            else:
                raise
    return []

 ## `web_search`

 The `@tool` decorator exposes this to the LangChain agent.
 The docstring is the description the LLM reads when deciding whether to call it.

 Only the first 3 results are sent to the LLM even if DDG returns 5
 — keeps context window usage reasonable.

In [ ]:
@tool
async def web_search(query: str) -> str:
    """
    Search the web and return relevant results.
    Use this tool for any current information, recent events,
    or facts you cannot answer with confidence.
    Arguments:
        query: The search request in natural language.
    """
    try:
        results = await asyncio.get_running_loop().run_in_executor(
            None, _ddg_search, query
        )

        if not results:
            return f"No results found for: {query}"

        formatted = [
            f"**{r.get('title', 'No title')}**\n"
            f"{r.get('body', '')}\n"
            f"Source: {r.get('href', '')}"
            for r in results[:3]
        ]

        log.info("web_search_done: query=%s results=%s", query[:50], len(results))
        return "\n\n---\n\n".join(formatted)  # fix: was "---n\n" (typo)

    except Exception as e:
        log.error("web_search_error: error=%s query=%s", str(e), query[:100])
        return f"Error during search: {str(e)}"